In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
import cv2
import torch
import easyocr
import numpy as np
from ultralytics import YOLO
from datetime import datetime
import time

print("All libraries loaded!")
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

All libraries loaded!
Using GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# Load YOLO model 
model = YOLO('yolov8n.pt')  # nano version 

print("✅ YOLO model loaded!")
print(f"Model running on: {'GPU' if torch.cuda.is_available() else 'CPU'}")

✅ YOLO model loaded!
Model running on: GPU


In [4]:
# Initialize EasyOCR
print("Loading EasyOCR... (first time takes a minute)")
reader = easyocr.Reader(['en'], gpu=True)
print("✅ EasyOCR loaded on GPU!")

Loading EasyOCR... (first time takes a minute)
✅ EasyOCR loaded on GPU!


In [5]:
def detect_vehicles(frame, model, reader):
    """Detects vehicles, reads plates and estimates speed"""
    
    # Run YOLO detection
    results = model(frame, device=0, verbose=False)
    
    # Vehicle classes in YOLO
    vehicle_classes = {
        2: 'Car',
        3: 'Motorcycle', 
        5: 'Bus',
        7: 'Truck'
    }
    
    detections = []
    
    for result in results:
        boxes = result.boxes
        
        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            
            # Only process vehicles with high confidence
            if class_id in vehicle_classes and confidence > 0.5:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                vehicle_type = vehicle_classes[class_id]
                
                detections.append({
                    'type': vehicle_type,
                    'confidence': confidence,
                    'bbox': (x1, y1, x2, y2)
                })
    
    return detections

print("✅ Detection function ready!")

✅ Detection function ready!


In [6]:
# Speed tracking
vehicle_tracker = {}
speed_tracker = {}

def calculate_speed(vehicle_id, bbox, fps=30):
    """Estimates vehicle speed based on bounding box movement"""
    
    cx = (bbox[0] + bbox[2]) // 2
    cy = (bbox[1] + bbox[3]) // 2
    
    if vehicle_id in vehicle_tracker:
        prev_cx, prev_cy, prev_time = vehicle_tracker[vehicle_id]
        
        # Pixel movement
        pixel_distance = np.sqrt((cx - prev_cx)**2 + (cy - prev_cy)**2)
        
        # Convert to estimated km/h (calibration factor)
        # This factor depends on camera height and angle
        speed_kmh = pixel_distance * 0.5 * fps * 0.036
        
        speed_tracker[vehicle_id] = round(speed_kmh, 1)
    
    vehicle_tracker[vehicle_id] = (cx, cy, time.time())
    
    return speed_tracker.get(vehicle_id, 0)


def draw_detections(frame, detections):
    """Draws bounding boxes and info on frame"""
    
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det['bbox']
        vehicle_type = det['type']
        confidence = det['confidence']
        speed = det.get('speed', 0)
        plate = det.get('plate', 'Reading...')
        
        # Color based on vehicle type
        colors = {
            'Car': (0, 255, 0),
            'Motorcycle': (255, 165, 0),
            'Bus': (0, 0, 255),
            'Truck': (255, 0, 255)
        }
        color = colors.get(vehicle_type, (0, 255, 0))
        
        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # Draw info background
        cv2.rectangle(frame, (x1, y1-80), (x2, y1), color, -1)
        
        # Draw text
        cv2.putText(frame, f"{vehicle_type} ({confidence:.0%})", 
                    (x1+5, y1-60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 2)
        cv2.putText(frame, f"Speed: {speed} km/h", 
                    (x1+5, y1-40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 2)
        cv2.putText(frame, f"Plate: {plate}", 
                    (x1+5, y1-20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 2)
    
    return frame

print("✅ Speed calculator and drawing functions ready!")

✅ Speed calculator and drawing functions ready!


In [7]:
def read_license_plate(frame, bbox, reader):
    """Crops vehicle region and reads license plate"""
    
    x1, y1, x2, y2 = bbox
    
    # Crop the vehicle region
    vehicle_crop = frame[y1:y2, x1:x2]
    
    if vehicle_crop.size == 0:
        return "N/A"
    
    # Focus on bottom half where plate usually is
    h = vehicle_crop.shape[0]
    plate_region = vehicle_crop[h//2:, :]
    
    # Read text with EasyOCR
    try:
        results = reader.readtext(plate_region, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
        
        if results:
            # Get the most confident result
            best_result = max(results, key=lambda x: x[2])
            text = best_result[1].upper()
            confidence = best_result[2]
            
            # Only return if confidence is high enough
            if confidence > 0.3 and len(text) >= 2:
                return text
    except:
        pass
    
    return "N/A"

print("✅ License plate reader ready!")

✅ License plate reader ready!


In [8]:
def run_vehicle_detector(source=0):
    """
    Main function - runs real-time vehicle detection
    source=0 means webcam
    source='video.mp4' means video file
    """
    
    cap = cv2.VideoCapture(source)
    
    if not cap.isOpened():
        print("❌ Could not open camera/video!")
        return
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_count = 0
    plate_update_interval = 10  # Read plate every 10 frames (performance)
    
    print("✅ Camera opened!")
    print("Press 'Q' to quit")
    print("-" * 40)
    
    plate_cache = {}  # Cache plates to avoid reading every frame
    
    while True:
        ret, frame = cap.read()
        
        if not ret:
            print("Video ended or camera disconnected")
            break
        
        frame_count += 1
        
        # Detect vehicles
        detections = detect_vehicles(frame, model, reader)
        
        # Process each detection
        for i, det in enumerate(detections):
            vehicle_id = i  # Simple ID based on position
            
            # Calculate speed
            speed = calculate_speed(vehicle_id, det['bbox'], fps)
            det['speed'] = speed
            
            # Read plate every N frames for performance
            if frame_count % plate_update_interval == 0:
                plate = read_license_plate(frame, det['bbox'], reader)
                plate_cache[vehicle_id] = plate
            
            det['plate'] = plate_cache.get(vehicle_id, 'Reading...')
        
        # Draw detections on frame
        frame = draw_detections(frame, detections)
        
        # Show FPS
        cv2.putText(frame, f"Vehicles detected: {len(detections)}", 
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        cv2.putText(frame, f"FPS: {fps:.0f}", 
                    (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        
        # Show frame
        cv2.imshow('Vehicle Detector 🚗', frame)
        
        # Press Q to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("✅ Detection stopped!")

print("✅ Main detection loop ready!")

✅ Main detection loop ready!


In [9]:
import os
print(os.getcwd())


C:\Users\Ayaan\vehicleDetector


In [10]:
run_vehicle_detector(source=r'C:\Users\Ayaan\vehicleDetector\traffic.mp4')

✅ Camera opened!
Press 'Q' to quit
----------------------------------------
Video ended or camera disconnected
✅ Detection stopped!


In [11]:
import cv2

cap = cv2.VideoCapture(r'C:\Users\Ayaan\vehicleDetector\traffic.mp4')
print(f"Video opened: {cap.isOpened()}")
print(f"Total frames: {cap.get(cv2.CAP_PROP_FRAME_COUNT)}")
print(f"FPS: {cap.get(cv2.CAP_PROP_FPS)}")
print(f"Width: {cap.get(cv2.CAP_PROP_FRAME_WIDTH)}")
print(f"Height: {cap.get(cv2.CAP_PROP_FRAME_HEIGHT)}")
cap.release()

Video opened: True
Total frames: 320.0
FPS: 30.0
Width: 2160.0
Height: 3840.0


In [12]:
run_vehicle_detector(source=r'C:\Users\Ayaan\vehicleDetector\traffic.mp4')

✅ Camera opened!
Press 'Q' to quit
----------------------------------------
Video ended or camera disconnected
✅ Detection stopped!


In [13]:
run_vehicle_detector(source=r'C:\Users\Ayaan\vehicleDetector\traffic.mp4')

✅ Camera opened!
Press 'Q' to quit
----------------------------------------
Video ended or camera disconnected
✅ Detection stopped!


In [ ]:
# Download car brand detection model
from ultralytics import YOLO
import urllib.request
import os

# Use a car make/model specific model
car_model = YOLO('yolov8n.pt')  # We'll use this as base

print("Checking available car models...")

# Test with a car classification model
car_classifier = YOLO('yolov8n-cls.pt')  # Classification model
print("✅ Car classifier downloaded!")